In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif, VarianceThreshold
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb
import optuna
import lightgbm as lgb
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

def get_morgan_fp(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp = generator.GetFingerprint(mol)
    return np.array(fp)



In [ ]:
df = pd.read_csv('../data/002_processed_data/xy_df_hopv.csv')
print(df.shape)
print(df['stability'].shape)
print(df['stability'].isna().sum())
df = df.dropna(subset=['stability'])
morgan_fps = np.array([get_morgan_fp(s) for s in df['smiles']])
print(morgan_fps.shape)  # (300, 1024)
print(df['stability'].isna().sum())
X = df[[column for column in df if column not in ['smiles', 'stability', 'homo', 'lumo']]]
y = df['stability']
le = LabelEncoder()



In [ ]:
# Combine raw features
X = np.hstack([X, morgan_fps])
print(f"Combined raw: {X.shape}")



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=999, stratify=y)

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
# Remove constant features before anything else
var_selector = VarianceThreshold(threshold=0)
X_train_var = var_selector.fit_transform(X_train)
print(f"Features after removing constants: {X_train_var.shape[1]}")  # Should be ~172

In [ ]:
# Remove constants first
var_selector = VarianceThreshold(threshold=0)
X_train_clean = var_selector.fit_transform(X_train)
X_test_clean = var_selector.transform(X_test)
imputer = SimpleImputer()
X_train_clean = imputer.fit_transform(X_train_clean)
X_test_clean = imputer.transform(X_test_clean)

# Then select best
selector = SelectKBest(f_classif, k=40)
X_train_selected = selector.fit_transform(X_train_clean, y_train)
X_test_selected = selector.transform(X_test_clean)

xgb_model = xgb.XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)

cv = StratifiedKFold(n_splits=5, random_state=999, shuffle=True)
accuracy = cross_val_score(xgb_model, X_train_selected, y_train, cv=cv, scoring='f1_macro').mean()
print(f'{accuracy:.3f}')

In [ ]:
# Train final model and evaluate on held-out test set
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=999)
xgb_model.fit(X_train_selected, y_train)


# Predictions
y_pred = xgb_model.predict(X_test_selected)

# Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Test F1 Macro: {f1_score(y_test, y_pred, average='macro'):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
data = pd.read_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv')
smiles = data['smiles']
morgan_fps = np.array([get_morgan_fp(s) for s in smiles])
morgan_fps

In [ ]:
descriptor_names = [x[0] for x in Descriptors._descList]


In [ ]:
def compute_descriptors(smiles_str):
    """
    WRITE A DESCRIPTION

    Parameters
    ----------
    smiles_str : str
        The SMILES string representation of a molecule.

    Returns
    -------
    tuple
        A tuple containing the values of all available molecular descriptors
        (217 descriptors in total, from RDKit descriptor list).
    """
    # YOUR CODE HERE
    mol = Chem.MolFromSmiles(smiles_str)
    features = tuple(getattr(Descriptors, name)(mol) for name in descriptor_names)
    return features


In [ ]:
X_kaggle = pd.DataFrame(smiles.apply(compute_descriptors).tolist(), columns=descriptor_names)
descriptors_features = X_kaggle.columns.tolist()
X_kaggle = np.hstack([X_kaggle, morgan_fps])
X_kaggle

In [ ]:
# Fit preprocessors on full data
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

var_selector = VarianceThreshold(threshold=0)
X_clean = var_selector.fit_transform(X_imputed)

y = le.transform(y)

selector = SelectKBest(f_classif, k=40)
X_selected = selector.fit_transform(X_clean, y)

xgb_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_selected, y)

X_test_imputed = imputer.transform(X_kaggle)
X_test_clean = var_selector.transform(X_test_imputed)
X_test_selected = selector.transform(X_test_clean)


In [ ]:
y_pred = xgb_model.predict(X_test_selected)
y_pred = le.inverse_transform(y_pred)
submission_df = pd.DataFrame({
    'smiles': smiles,
    'stability': y_pred
})
pd.DataFrame(submission_df).to_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv', index=False)


In [ ]:
# Assuming you have both models' predictions on the Kaggle test set

old_preds = pd.read_csv('../data/003_kaggle_competition_data/25261_classification_sample_submission.csv')['stability']
new_preds = y_pred

# Compare strings directly
diff_count = (old_preds != new_preds).sum()
print(f"Different predictions: {diff_count} / {len(old_preds)} ({100*diff_count/len(old_preds):.1f}%)")

# Which ones differ?
diff_idx = np.where(old_preds != new_preds)[0]
print(f"Indices where they differ: {diff_idx}")

# Class distributions
from collections import Counter
print(f"\nOld model: {Counter(old_preds)}")
print(f"New model: {Counter(new_preds)}")

In [ ]:
# Get selected feature mask
selected_mask = selector.get_support()

# Count how many from RDKit vs Morgan
n_rdkit = 217
n_rdkit_selected = selected_mask[:n_rdkit].sum()
n_morgan_selected = selected_mask[n_rdkit:].sum()

print(f"RDKit features selected: {n_rdkit_selected}/217")
print(f"Morgan features selected: {n_morgan_selected}/1024")